# Lab 00 — Setup & orientation

**Goal:** connect to your Microsoft Foundry project with `DefaultAzureCredential`, verify your environment, and take a first look at the Control Plane.

## What is the Foundry Control Plane?

The Control Plane gives IT and platform teams a **fleet-wide view** of everything running on Microsoft Foundry — agents, models, tools, evaluations, guardrails, quotas, and cost — and lets them enforce enterprise policy without slowing developers down.

It surfaces five panes (all under the **Operate** button in the [Foundry portal](https://ai.azure.com)):

| Pane | Use |
|------|-----|
| **Overview** | Snapshot of health, activity, and risk across the fleet |
| **Assets** | Discover agents, models, evaluations, connections, projects |
| **Compliance** | Guardrail policies, content safety, red-team results |
| **Quota** | Model quota and utilization |
| **Admin** | RBAC, networking, allowed models |

In these labs you'll play the role of the **Zava platform team** — governing a small fleet of retail agents.

> Reference: [What is Microsoft Foundry Control Plane?](https://learn.microsoft.com/en-us/azure/foundry/control-plane/overview)

## 1. Sign in to Azure

Open a terminal and run:

```powershell
az login
az account set --subscription "<your-subscription-id>"
```

`DefaultAzureCredential` will pick up your `az` session automatically — no secrets in code.

## 2. Load environment variables

Copy `.env.example` (in the repo root) to `.env` and fill in at minimum:

- `FOUNDRY_PROJECT_ENDPOINT`
- `FOUNDRY_MODEL_NAME`
- `AZURE_SUBSCRIPTION_ID`
- `AZURE_RESOURCE_GROUP`

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the repo root (two levels up from this notebook)
repo_root = Path.cwd().parent
env_path = repo_root / ".env"
load_dotenv(env_path, override=True)

print(f"Loaded env from: {env_path}")

REQUIRED = [
    "FOUNDRY_PROJECT_ENDPOINT",
    "FOUNDRY_MODEL_NAME",
    "AZURE_SUBSCRIPTION_ID",
    "AZURE_RESOURCE_GROUP",
]
missing = [k for k in REQUIRED if not os.getenv(k)]
if missing:
    print(f"[!] Missing env vars: {missing}")
else:
    print("[OK] All required env vars are set.")

## 3. Authenticate with `DefaultAzureCredential`

`DefaultAzureCredential` tries several sources in order: env vars, managed identity, Azure CLI, VS Code, etc. For local dev, the Azure CLI branch (`az login`) is what activates.

In [ ]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)

# Force a token acquisition to confirm auth works
token = credential.get_token("https://management.azure.com/.default")
print(f"[OK] Got management-plane token, expires in {int((token.expires_on - __import__('time').time())/60)} min")

## 4. Connect to the Foundry project

In [ ]:
from azure.ai.projects import AIProjectClient

project = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=credential,
)

print(f"[OK] Connected to project endpoint:\n    {os.environ['FOUNDRY_PROJECT_ENDPOINT']}")

## 5. Sanity check — list deployments

If this returns your deployed chat model, you're wired up correctly.

In [ ]:
from rich.table import Table
from rich.console import Console

console = Console()

table = Table(title="Model deployments in this Foundry project")
table.add_column("Name")
table.add_column("Model")
table.add_column("Type")

count = 0
try:
    for d in project.deployments.list():
        table.add_row(
            getattr(d, "name", ""),
            getattr(d, "model_name", getattr(d, "model", "")),
            getattr(d, "type", getattr(d, "deployment_type", "")),
        )
        count += 1
except Exception as e:
    console.print(f"[yellow]Could not list deployments: {e}[/yellow]")

console.print(table)
print(f"Total deployments: {count}")

## 6. Quick call to your chat model

One tiny inference call to confirm end-to-end plumbing.

In [ ]:
chat = project.get_openai_client()

response = chat.chat.completions.create(
    model=os.environ["FOUNDRY_MODEL_NAME"],
    messages=[
        {"role": "system", "content": "You are the Zava support bot. Be concise."},
        {"role": "user", "content": "In one sentence: what is Zava?"},
    ],
    max_completion_tokens=600,
)
print(response.choices[0].message.content)

## 7. See it in the portal

Now open the [Foundry portal](https://ai.azure.com) and take a look:

1. Open your project.
2. Click **Operate** in the upper-right toolbar.
3. Land on **Overview** — you'll see aggregate signals across the fleet.
4. Click **Assets** — everything you just enumerated appears here plus much more (agents, evaluations, connections).

You now have the plumbing to run every remaining lab. Head to [`01-fleet-inventory.ipynb`](01-fleet-inventory.ipynb).